## Setup do Ambiente
- Importação de bibliotecas essenciais do PySpark
- Definição dos paths utilizados para as tabelas
- Utilização do catálogo `catalogo`, schemas `silver_db_name`, `gold_db_name`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    explode,
    sequence,
    col,
    year,
    quarter,
    month,
    weekofyear,
    dayofmonth,
    dayofweek,
    when
)
from pyspark.sql.types import DateType


### Definição de Variáveis Globais
- Centralizamos os nomes de catálogos, bancos de dados e caminhos.
- **Boas Práticas:** Evitar "hardcoding" (escrever o caminho diretamente no código várias vezes). Se o nome do catálogo mudar no futuro, alteramos apenas aqui.

In [0]:
catalogo = "medalhao_credit"
silver_db_name = "silver_credit"
gold_db_name = "gold_credit"

In [0]:
spark.sql(f"USE CATALOG {catalogo};")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_db_name};")
spark.sql(f"USE SCHEMA {gold_db_name};")

## Criação da tabela `dm_tempo`

Colunas da tabela `dm_tempo`:

- `sk_tempo`
- `ano`
- `trimestre`
- `mes`
- `semana_do_ano`
- `dia`
- `dia_da_semana_num`
- `dia_da_semana_nome`
- `mes_nome`
- `eh_fim_de_semana`

In [0]:
data_inicio = '2016-01-01'
data_fim = '2019-01-01'

df_datas = (
    spark.createDataFrame([(data_inicio, data_fim)], ['data_inicio', 'data_fim'])
    .select(explode(sequence(col('data_inicio').cast(DateType()), col('data_fim').cast(DateType()))).alias('sk_tempo'))
    .withColumn('ano', year(col('sk_tempo')))
    .withColumn('trimestre', quarter(col('sk_tempo')))
    .withColumn('mes', month(col('sk_tempo')))
    .withColumn('semana_do_ano', weekofyear(col('sk_tempo')))
    .withColumn('dia', dayofmonth(col('sk_tempo')))
    .withColumn('dia_da_semana_num', dayofweek(col('sk_tempo')))
    .withColumn('dia_da_semana_nome', 
        when(col('dia_da_semana_num') == 1, 'Domingo')
        .when(col('dia_da_semana_num') == 2, 'Segunda-feira')
        .when(col('dia_da_semana_num') == 3, 'Terça-feira')
        .when(col('dia_da_semana_num') == 4, 'Quarta-feira')
        .when(col('dia_da_semana_num') == 5, 'Quinta-feira')
        .when(col('dia_da_semana_num') == 6, 'Sexta-feira')
        .when(col('dia_da_semana_num') == 7, 'Sabado')
    )
    .withColumn('mes_nome',
        when(col('mes') == 1, 'Janeiro')
        .when(col('mes') == 2, 'Fevereiro')
        .when(col('mes') == 3, 'Março')
        .when(col('mes') == 4, 'Abril')
        .when(col('mes') == 5, 'Maio')
        .when(col('mes') == 6, 'Junho')
        .when(col('mes') == 7, 'Julho')
        .when(col('mes') == 8, 'Agosto')
        .when(col('mes') == 9, 'Setembro')
        .when(col('mes') == 10, 'Outubro')
        .when(col('mes') == 11, 'Novembro')
        .when(col('mes') == 12, 'Dezembro')
    )
    .withColumn('eh_fim_de_semana', when(col('dia_da_semana_num').isin([1,7]), 'Sim').otherwise('Não'))
)

df_datas.limit(20).display()

df_datas.write.mode('overwrite').saveAsTable(f'{catalogo}.{gold_db_name}.dm_tempo')

## Criação da tabela `ft_chamados`

Colunas da tabela `ft_chamados`:

- `id_chamado`
- `id_cliente`
- `id_atendente`
- `motivo`
- `canal`
- `resolvido`
- `nota_atendimento`
- `categoria_nota`
- `status_canal`
- `valor_custo`

In [0]:
df_chamados_geral = spark.table(f'{catalogo}.{silver_db_name}.ft_chamados_geral')
display(df_chamados_geral.limit(20))

In [0]:
print(f"Colunas de {catalogo}.{silver_db_name}.ft_chamados_geral:\n")
print(f"{df_chamados_geral.columns}\n")
print(f"Schema de {catalogo}.{silver_db_name}.ft_chamados_geral:\n")
df_chamados_geral.printSchema()

In [0]:
df_gold_ft_chamados = df_chamados_geral.select(
    col('id_chamado'), col('id_cliente'),     
    when(col('id_atendente') == -1, None).otherwise(col('id_atendente')).alias('id_atendente'), 
    col('motivo'), col('canal'), col('status_canal'), col('resolvido'), col('nota_atendimento'), col('categoria_nota'), col('valor_custo')
)
df_gold_ft_chamados.limit(15).display()
df_gold_ft_chamados.printSchema()

In [0]:
df_gold_ft_chamados.write.mode('overwrite').saveAsTable(f'{catalogo}.{gold_db_name}.ft_chamados')

# Criação da tabela ft_clientes

## Releitura e Garantia de Contexto
Para assegurar a disponibilidade dos dados na sessão atual e mitigar eventuais perdas de referência em memória, realizamos a leitura explícita das tabelas `ft_chamados_geral` e `ft_clientes`.
Essa ação garantiu que os DataFrames estivessem atualizados e prontos para o cruzamento final, prevenindo erros de execução nas células subsequentes.

In [0]:
df_chamados_silver = spark.table(f"{catalogo}.{silver_db_name}.ft_chamados_geral")

In [0]:
df_clientes_silver = spark.table(f"{catalogo}.{silver_db_name}.ft_clientes")

## Cálculo de Métricas Comportamentais do Cliente

Nesta etapa, consolidamos o histórico transacional de cada cliente em métricas de comportamento agregadas. Agrupamos os dados da tabela `df_chamados_silver` pela chave `id_cliente` e calculamos seis indicadores fundamentais para compor o perfil do consumidor:

1.  **Volumetria:** Contagem total de chamados realizados.
2.  **Financeiro:** Soma do custo operacional gerado pelo cliente.
3.  **Satisfação:** Média das notas de atendimento atribuídas.
4.  **Esforço:** Tempo total acumulado que o cliente passou em espera.
5.  **Recência:** Identificação da data do último contato para análises de retenção.
6.  **Resolutividade:** Cálculo percentual da taxa de sucesso nas solicitações do cliente.

Essa agregação transforma dados granulares de chamados em uma visão única de "valor e comportamento" por cliente.

In [0]:
# "Fatos" do cliente
df_metrics_comportamento = (
    df_chamados_silver
    .groupBy("id_cliente")
    .agg(
        # 1. Volumetria
        F.count("id_chamado").alias("total_chamados_historico"),
        
        # 2. Financeiro (Custo que o cliente gerou para a operação)
        F.round(F.sum("valor_custo"), 2).alias("custo_total_atendimento"),
        
        # 3. Satisfação (Média das notas dele)
        F.round(F.avg("nota_atendimento"), 1).alias("nota_media_satisfacao"),
        
        # 4. Tempo (Quanto tempo ele já perdeu com a gente)
        F.sum("tempo_espera_segundos").alias("tempo_total_espera_seg"),
        
        # 5. Recência (Quando foi a última vez que ele ligou?)
        F.max("hora_abertura_chamado").alias("data_ultimo_contato"),
        
        # 6. Resolutividade (Quantos % dos problemas dele foram resolvidos)
        F.round(
            (F.sum(F.when(F.col("resolvido") == "Sim", 1).otherwise(0)) / F.count("id_chamado")) * 100, 
            2
        ).alias("taxa_resolucao_pessoal")
    )
)

display(df_metrics_comportamento.limit(5))

## Enriquecimento e Segmentação Demográfica do Cliente

Nesta etapa, isolamos os dados cadastrais essenciais da tabela `df_clientes_silver` (como nome, e-mail e região) para compor a dimensão de perfil.

Para potenciar as análises de Marketing na camada Gold, enriquecemos esta base criando a coluna `faixa_etaria`. Aplicamos regras de negócio condicionais para segmentar os clientes em quatro grandes grupos ("Jovem", "Adulto Jovem", "Adulto" e "Senior"), transformando a variável numérica contínua (`idade`) numa variável categórica pronta para filtros e agrupamentos em *dashboards*.

In [0]:
# Tratamento da Dimensão Cliente (Dados Cadastrais)
df_perfil_cliente = (
    df_clientes_silver
    .select("id_cliente", "nome_cliente", "email_cliente", "regiao", "idade")
    
    # Criando Faixa Etária (Facilita análise de Marketing)
    .withColumn("faixa_etaria", 
                F.when(F.col("idade") < 25, "Jovem (ate 24)")
                .when(F.col("idade").between(25, 40), "Adulto Jovem (25-40)")
                .when(F.col("idade").between(41, 60), "Adulto (41-60)")
                .otherwise("Senior (+60)")
    )
)

display(df_perfil_cliente.limit(5))

## Consolidação da Visão 360º do Cliente

Nesta etapa crítica, nós unificamos as dimensões de perfil e comportamento para criar a tabela final. Realizamos um *Left Join* entre a base de clientes e as métricas calculadas, garantindo que mesmo os clientes "silenciosos" (sem histórico de chamados) fossem mantidos na análise.

Para sanear os dados, aplicamos a imputação de zero nos valores nulos resultantes da junção (para métricas de volume e custo). Em seguida, implementamos uma lógica de segmentação estratégica na coluna `perfil_cliente`, classificando os consumidores em categorias como "Detrator Crítico", "Promotor" ou "Neutro" com base na combinação da volumetria de contato e da satisfação média. Por fim, selecionamos as colunas definitivas e adicionamos a data de processamento.

In [0]:
df_gold_cliente_360 = (
    df_perfil_cliente.alias("cli")
    # Left Join: todos os clientes, mesmo os que nunca ligaram
    .join(df_metrics_comportamento.alias("fat"), "id_cliente", "left")
    
    # Tratamento de Nulos 
    .fillna(0, subset=["total_chamados_historico", "custo_total_atendimento", "tempo_total_espera_seg"])
    
    .withColumn("perfil_cliente",
                F.when((F.col("total_chamados_historico") > 5) & (F.col("nota_media_satisfacao") < 5), "DETRATOR CRITICO")
                .when((F.col("nota_media_satisfacao") >= 9), "PROMOTOR")
                .when(F.col("total_chamados_historico") == 0, "SILENCIOSO (SEM CONTATO)")
                .otherwise("NEUTRO / ATIVO")
    )
    
    .select(
        "id_cliente",
        "nome_cliente",
        "email_cliente",
        "regiao",
        "idade",
        "faixa_etaria",
        "total_chamados_historico",
        "custo_total_atendimento",
        "nota_media_satisfacao",
        "taxa_resolucao_pessoal",
        "data_ultimo_contato",
        "perfil_cliente",
        F.current_timestamp().alias("data_processamento_gold")
    )
)

display(df_gold_cliente_360)

## Persistência da Tabela Fato de Perfil do Cliente

Como etapa conclusiva deste fluxo, nós materializamos o DataFrame consolidado `df_gold_cliente_360` no armazenamento físico. Definimos o destino como `ft_cliente_perfil` dentro do esquema Gold.

Utilizamos o formato Delta e configuramos o modo de escrita como `overwrite` (sobrescrita), juntamente com a opção `overwriteSchema`. Essa decisão assegura que a tabela reflita sempre a versão mais atualizada e completa do perfil dos clientes, permitindo inclusive alterações na estrutura dos dados (evolução de esquema) sem causar falhas no processo.

In [0]:
tabela_destino = f"{catalogo}.{gold_db_name}.ft_clientes"

(
    df_gold_cliente_360.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_destino)
)

print(f"Tabela Gold (Customer 360) criada com sucesso em: {tabela_destino}")

## Faixa etária dos clientes X etapa em que resolveram o problema

## Leitura dos Dados para Análise de Funil

Para dar início à construção da visão estratégica de "Faixa Etária x Etapa de Resolução", nós carregamos a tabela `ft_chamados_geral` do catálogo. Essa leitura garante que temos em memória o histórico completo e granular dos atendimentos, o que é indispensável para classificar corretamente cada interação segundo o canal utilizado e o perfil do cliente nas etapas seguintes.

In [0]:
df_chamados_geral = spark.table(f"{catalogo}.{silver_db_name}.ft_chamados_geral")

## Modelagem de Etapas do Funil e Demografia

Nesta transformação, nós recriamos a categorização de `faixa_etaria` para assegurar a consistência dos dados demográficos na análise. Simultaneamente, introduzimos uma nova dimensão estratégica chamada `etapa_funil`.

Mapeamos os canais de atendimento numa hierarquia lógica de resolução:
* **Autosserviço:** Agrupamos canais digitais e automatizados (Chatbot, URA, Web, App).
* **Nível 1:** Atendimento humano generalista (Atendimento Inicial).
* **Nível 2:** Atendimento humano especialista (Atendimento Especializado).

Essa estruturação é fundamental para analisarmos posteriormente a eficiência da retenção digital versus o transbordo para o atendimento humano.

In [0]:
df_transformado = (
    df_chamados_geral
    
    # 1. Recriar Faixa Etária (Caso não tenha trazido do join anterior)
    .withColumn("faixa_etaria", 
                F.when(F.col("idade") < 25, "1. Jovem (ate 24)")
                .when(F.col("idade").between(25, 40), "2. Adulto Jovem (25-40)")
                .when(F.col("idade").between(41, 60), "3. Adulto (41-60)")
                .when(F.col("idade") > 60, "4. Senior (+60)")
                .otherwise("5. Nao Informado")
    )
    
    # 2. Definir Etapa do Funil (Onde o chamado parou)
    .withColumn("etapa_funil",
                F.when(F.col("canal").isin("Chatbot", "URA", "Web", "App"), "1. Autosservico (Digital/Robo)")
                .when(F.col("canal") == "Atendimento Inicial", "2. Nivel 1 (Humano Generalista)")
                .when(F.col("canal") == "Atendimento Especializado", "3. Nivel 2 (Especialista)")
                .otherwise("4. Outros")
    )
)

display(df_transformado.select("id_chamado", "canal", "etapa_funil", "idade", "faixa_etaria").limit(5))

## Cálculo de Eficiência do Funil por Faixa Etária

Nesta etapa analítica, nós consolidamos os dados agrupando-os por duas dimensões chave: `faixa_etaria` e `etapa_funil`.

Dentro da agregação, calculamos KPIs fundamentais para entender a performance do atendimento:
* **Volumetria:** O total de tentativas de contato.
* **Sucesso:** O volume absoluto de chamados resolvidos.
* **Esforço:** O Tempo Médio de Atendimento (TMA) em segundos.
* **Qualidade:** A nota média de satisfação atribuída.

Além disso, derivamos matematicamente a `taxa_resolucao_pct`. Essa métrica é crucial para comparar a eficácia relativa de cada canal para cada perfil demográfico, permitindo identificar, por exemplo, se determinados grupos têm maior dificuldade com canais digitais. Por fim, ordenamos o resultado para estruturar a visualização lógica do funil.

In [0]:
df_view_funil_idade = (
    df_transformado
    .groupBy("faixa_etaria", "etapa_funil")
    .agg(
        # Volumetria Total
        F.count("id_chamado").alias("total_tentativas"),
        
        # Quantos foram resolvidos?
        F.sum(F.when(F.col("resolvido") == "Sim", 1).otherwise(0)).alias("total_resolvidos"),
        
        # Tempo Médio gasto nessa etapa
        F.round(F.avg("tempo_atendimento_segundos"), 0).alias("tma_medio_seg"),
        
        # Nota média dada nessa etapa
        F.round(F.avg("nota_atendimento"), 2).alias("nota_media")
    )
    
    # Cálculo da Taxa de Resolução (Eficácia do Funil)
    .withColumn("taxa_resolucao_pct", 
                F.round((F.col("total_resolvidos") / F.col("total_tentativas")) * 100, 2))
    
    # Ordenação para o gráfico (Do Jovem pro Senior, do Topo pro Fundo)
    .orderBy("faixa_etaria", "etapa_funil")
)

display(df_view_funil_idade)

In [0]:
tabela_destino = f"{catalogo}.{gold_db_name}.funil_resolucao_etaria"

(
    df_view_funil_idade.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_destino)
)

print(f"View Gold criada com sucesso: {tabela_destino}")